# Assignment 2: Language Modeling

## Overview

Instruction-tuning is a supervised fine-tuning technique that trains language models on a dataset of <instruction, output> pairs to improve their ability to follow user prompts. The goal of this assignment is to perform instruction-tuning for a less capable model, i.e., Qwen1.5-0.5B, using a instruction dataset synthetically generated by a more capable model (GPT-5.2).

Let a training example consist of a prompt $x$ (instruction) and a response $y$, tokenized into:

$$x = (x_1, \ldots, x_m), \quad y = (y_1, \ldots, y_n)$$

Concatenated into the full input sequence:

$$s = (s_1, \ldots, s_T) = (x_1, \ldots, x_m, y_1, \ldots, y_n), \quad T = m + n$$

An auto-regressive language model with parameters $\theta$ computes the token-level cross-entropy loss:

$$\mathcal{L}_\theta = -\log P(s_t \mid s_{<t})$$

The dataset contains **1,097 synthetic `<instruction, output>` pairs**, split **70 / 15 / 15** into train, dev, and test sets.

## Requirements

- Implement each task in the cells marked **TODO: your code here**
- You must implement the training and evaluation using plain PyTorch only. Frameworks such as HuggingFace `transformers`, `trl`, or others are **not allowed**.
- The tokenizer and model may be loaded via HuggingFace `transformers`\ `peft`.
- Save your instruction-tuned models as **you will need them for Assignment 3**.

## Submission
- Submit only the .ipynb file. No need to submit any models.

## Deadline
**April 2, 23:59 CET**

## Note: GPU Resources

**Test your code on CPU first.** Work on CPU first to make sure everything runs before switching to GPU for training. GPU time is limited — do not waste it debugging code that could be verified on CPU.



## Install dependencies

In [1]:
!pip install transformers datasets torch accelerate huggingface_hub tqdm peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 14.3 MB/s  0:00:00


## Imports & device setup

In [1]:
import math
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from datasets import load_dataset, DatasetDict
from tqdm import tqdm

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu" # Laptop
#DEVICE = "cuda" if torch.cuda.is_available() else "cpu" # Home

print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps


## Load the instruction-tuning dataset
The dataset is provided as a JSONL file. We convert it to a *HuggingFace Dataset* for easier access and processing.

In [2]:
dataset = load_dataset("json", data_files={
    # Laptop paths
    "train": r"/Users/merterol/Desktop/uzh_code/uzh/Computational Linguistics/Master/RAG/Assignment 2/train.jsonl",
    "dev":   r"/Users/merterol/Desktop/uzh_code/uzh/Computational Linguistics/Master/RAG/Assignment 2/dev.jsonl",
    "test":  r"/Users/merterol/Desktop/uzh_code/uzh/Computational Linguistics/Master/RAG/Assignment 2/test.jsonl",
    
    # Home paths
})

# Inspect dataset structure
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'metadata'],
        num_rows: 767
    })
    dev: Dataset({
        features: ['instruction', 'input', 'output', 'metadata'],
        num_rows: 164
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'metadata'],
        num_rows: 166
    })
})


In [11]:
# Inspect a single training example
print(dataset["train"][0])

{'instruction': 'Explain the answer to the question below based strictly on the provided document.', 'input': 'Question:\nhas anyone investigated the effect of shock generated vorticity on heat\ntransfer to a blunt body .\n\nDocument:\nheat transfer at the forward stagnation point of blunt\nbodies .\nheat transfer at the forward stagnation point of blunt\nbodies .\n  relations are presented for the calculation of heat transfer at\nthe forward stagnation point of both two-dimensional and axially\nsymmetric blunt bodies .  the relations for the heat transfer, which were\nobtained from exact solutions to the equations of the laminar boundary\nlayer, are presented in terms of the local velocity gradient at the\nstagnation point .  these exact solutions include effects of variation\nof fluid properties, prandtl number, and transpiration cooling .\nexamples illustrating the calculation procedure are also included .', 'output': 'The provided document does not mention any investigation of **sh

## Task 0: Prompt Formatting

Before tokenization, each data sample is formatted as a single sequence by concatenating the instruction, input, and response with section headers.

**Note**
1. Section headers (e.g., `### Instruction:`, `### Input:`, `### Response:`) act as structural delimiters. The model learns to associate these markers with the roles of different segments and to generate the response following the ### Response: header.
2. The template must be identical at inference time


In [3]:
def format_prompt(sample: dict) -> tuple[str, str]:
    """
    Convert a raw dataset sample into a (prompt, response) string pair.

    Requirements:
        - Combine the 'instruction' and 'input' fields into a prompt string
            using the following fixed template
        - The prompt must end with the '### Response:\n' header and no response text.
        - Take the 'output' field directly as the response string y.
        - A sample complete template to use:
                ### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response:\n{Response}
        - The same template must be used at inference time with exactly the section headersand and newlines and {response} being generated.

    Args:
        sample: A single dataset sample with keys 'instruction', 'input',
                 and 'output'.

    Returns:
        A tuple (prompt, response) where:
            prompt   : str  — the full instruction string x, ending with
                              the response header and no response text.
            response : str  — the raw output string y.

    Note:
        The prompt and response are returned separately as a tuple to identify the
        prompt boundary m, which is required for instruction-token masking (Task 1)
        and response-only perplexity (Task 3).
    """
    # DONE: your code here
    input_text = sample['input']
    output = sample['output']
    instruct = sample['instruction']
    
    prompt = f"### Instruction:\n{instruct}\n\n### Input:\n{input_text}\n\n### Response:\n"
    response = f"{output}"
    
    return prompt, response

## Load the tokenzier

In [4]:
# Since the same tokenizer is reused across tasks (Task 1, 2, and 3), we load it once here at the top rather than repeating it in each task.
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-0.5B")

# Qwen tokenizer may not have a pad token — reuse eos
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


## Task 1: Completion-only tuning

Fine-tune using **only response tokens** $y$ by masking the instruction tokens in the labels:

$$\mathcal{L}_\theta = -\frac{1}{n} \sum_{i=1}^{n} \log P(y_i \mid x_{1:m},\, y_{<i})$$


### Task 1a: Tokenization/ Data preparation for instruction masking

PyTorch expects all inputs to be tensors of fixed length. We therefore tokenize each formatted sequence $s = (x, y)$ into `input_ids`, pad to `MAX_LENGTH`, and construct an `attention_mask` to ignore padding tokens and a `labels` tensor for the loss computation.

In [9]:
def tokenize_completion_only(sample: dict, tokenizer, max_length: int) -> dict[str, list[int]]:
    """
    Tokenize one dataset sample for Task 1 (completion-only tuning).

    The full token sequence s = [x; y] is constructed by concatenating
    the prompt tokens x (length m) and response tokens y (length n),
    then padded to max_length. The labels tensor mirrors input_ids but
    masks all instruction and padding positions with -100 so that the
    cross-entropy loss is computed only over response tokens y.

    Args:
        sample: A single dataset record with keys 'instruction', 'input',
                 and 'output'.
        tokenizer: The tokenizer to use for encoding the text.
        max_length: Maximum sequence length for truncation and padding.

    Returns:
        A dict with three equal-length lists (length is max_length):
            input_ids      – token ids for the full sequence [x; y],
                             right-padded with pad_token_id.
            attention_mask – 1 for real tokens, 0 for padding positions.
            labels         – token ids for response positions only;
                             -100 everywhere else (ignored by the loss).

    """
    # TODO: your code here
    prompt, response = format_prompt(sample)
    tokens = tokenizer(prompt + response, truncation=True, max_length=max_length, padding="max_length")
    input_ids = tokens["input_ids"]
    attention_mask = tokens["attention_mask"]
    labels = []
    
    for i in range(len(input_ids)):
        if attention_mask[i] == 0 or i < len(tokenizer(prompt)["input_ids"]):
            labels.append(-100)
        else:
            labels.append(input_ids[i])
            
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

### Data Preparation

In [10]:

#TODO: Set appropritate MAX_LENGTH
# Hint: inspect sequence lengths in the training set. Must be accomodations of both prompt and response tokens.
MAX_LENGTH = 512

# Prepare a dataset for training with input ids, attention mask and labels
remove_cols = dataset["train"].column_names
tokenized_completion = DatasetDict({
    split: dataset[split].map(
        tokenize_completion_only,
        fn_kwargs={"tokenizer": tokenizer, "max_length": MAX_LENGTH},
        remove_columns=remove_cols,
    )
    for split in ("train", "dev")
})

# Change to tensors
tokenized_completion.set_format("torch")
print(tokenized_completion)

Map: 100%|██████████| 164/164 [00:17<00:00,  9.64 examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 767
    })
    dev: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 164
    })
})


### Task 1b: Load the model and apply a LoRA adapter

Implement `load_model` below to load the base model and wrap it with a LoRA adapter ([PEFT docs](https://huggingface.co/docs/peft)) before training. The same `load_model` will be reused for Task 2

**Why LoRA?**

This dataset contains only ~767 training examples. Fine-tuning all 500 M+ parameters of Qwen1.5-0.5B on such a small set causes **overfitting**: the model memorises training sequences but generalises poorly, visible as a widening gap between `train_loss` and `dev_loss`.

[LoRA (Low-Rank Adaptation)](https://huggingface.co/docs/peft/main/en/conceptual_guides/lora) addresses this by **freezing** the original weights and injecting small trainable low-rank matrices into selected projection layers:

$$W' = W_0 + BA, \quad B \in \mathbb{R}^{d \times r},\; A \in \mathbb{R}^{r \times k}, \quad r \ll \min(d,\, k)$$

Only the delta $\Delta W = BA$ is trained, reducing trainable parameters from ~500 M to ~3.8 M (&lt;1 %). This acts as an implicit regulariser that makes fine-tuning feasible on a small dataset.

**Key LoRA hyperparameters**

| Parameter | Description |
|-----------|-------------|
| `r` | Rank of the low-rank matrices — controls adapter capacity |
| `lora_alpha` | Scaling factor applied to the LoRA output ($\alpha / r$) |
| `target_modules` | Which projection layers receive LoRA adapters |
| `lora_dropout` | Dropout applied to LoRA activations for regularisation |

After training, call `model.merge_and_unload()` to fold the LoRA weights back into the base model and return a standard HuggingFace checkpoint with no PEFT overhead.

In [11]:
def load_model(model_name: str = "Qwen/Qwen1.5-0.5B") -> AutoModelForCausalLM:
    """
    Load a pre-trained causal language model and wrap it with a LoRA adapter.

    Refer to the Task 1b description above for guidance on which LoRA
    hyperparameters to use and which projection layers to target.
    See the PEFT docs linked there for the LoraConfig API.

    Args:
        model_name: HuggingFace model ID to load (default "Qwen/Qwen1.5-0.5B").

    Returns:
        A PEFT-wrapped AutoModelForCausalLM ready for training.
    """
    # TODO: your code here
    model = AutoModelForCausalLM.from_pretrained(model_name)
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none"
    )
    
    peft_model = get_peft_model(model, lora_config)
    
    return peft_model


### Task 1c: Define the PyTorch training loop

You are allowed to add any other hyperparameters you wish to implement with plain PyTorch. The same `train_model` function will be used for Task 2.

Examples of additional hyperparameters you may consider (**Not mandatory**):
- **`lr_scheduler`**: Learning rate schedule such as cosine annealing or linear decay (e.g., `torch.optim.lr_scheduler.CosineAnnealingLR`)
- **`max_grad_norm`**: Gradient clipping threshold to stabilize training (e.g., `torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm=1.0)`)
- **`weight_decay`**: L2 regularization strength passed to AdamW (e.g., `weight_decay=0.1`)
- **`accumulation_steps`**: Number of steps to accumulate gradients before updating, simulating a larger batch size (Effective batch size = batch_size × accumulation_steps. For example, `accumulation_steps=4` with `batch_size=4`, effective batch size = 16). This is useful for avoiding CUDA out-of-memory (OOM) errors when larger batches don’t fit in GPU memory.

In [14]:
# You can change the function definition to add more params if you wish so
def train_model(
    model: AutoModelForCausalLM,
    tokenized_dataset: dict,
    epochs: int = 3,
    batch_size: int = 8,
    lr: float = 2e-5,
) -> AutoModelForCausalLM:
    """
    Fine-tune a causal language model using a plain PyTorch training loop.

    Args:
        model:             A pre-loaded AutoModelForCausalLM instance to fine-tune.
        tokenized_dataset: A DatasetDict with 'train' and 'dev' splits, each
                           containing tensors: input_ids, attention_mask, labels.
        epochs:            Number of full passes over the training set (default 3).
        batch_size:        Number of examples per gradient update step (default 8).
        lr:                Peak learning rate for AdamW (default 2e-5).

    Returns:
        The fine-tuned model (modified in-place; returned for convenience).

    
    Requirements:
        - The training loop MUST include both a training phase and a dev evaluation
          phase for every epoch.
        - After each epoch, you MUST print both train_loss and dev_loss.   


    Note:
        1. Initialize an AdamW optimizer with weight decay.
        2. Create DataLoaders for the train and dev splits.
        3. For each epoch:
            a. Training loop:
                - Forward pass to get raw logits.
                - Apply the causal shift: logit at position t predicts token t+1.
                - Compute cross-entropy loss, ignoring positions where labels == -100. 
                  (Padding and masked positions are ignored via ignore_index=-100 in F.cross_entropy.)
                - Backward pass and optimizer step.
            b. Evaluation loop (no gradients):
                - Compute dev loss using the same procedure.
            c. Print metrics after each epoch:
                  train_loss, dev_loss


    """

    # TODO: your code here
    model.to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    data_loader_train = DataLoader(tokenized_dataset["train"], batch_size=batch_size, shuffle=True)
    data_loader_dev = DataLoader(tokenized_dataset["dev"], batch_size=batch_size)
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        
        for batch in tqdm(data_loader_train, desc=f"Epoch {epoch+1}/{epochs} - Training"):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(data_loader_train)
        
        model.eval()
        dev_loss = 0.0
        
        with torch.no_grad():
            for batch in tqdm(data_loader_dev, desc=f"Epoch {epoch+1}/{epochs} - Evaluating"):
                input_ids = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)
                labels = batch["labels"].to(DEVICE)
                
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss
                dev_loss += loss.item()
        
        avg_dev_loss = dev_loss / len(data_loader_dev)
        
        print(f"Epoch {epoch+1}/{epochs} - Average Train Loss: {avg_train_loss:.4f}, Average Dev Loss: {avg_dev_loss:.4f}")
    
    return model

### Task 1d: Fine-tune and save the completion-only model

In [15]:
model_completion = load_model()
# TODO: Call train_model with the hyperparameters you chose
model_completion = train_model(model_completion, tokenized_completion)

# Merge LoRA weights into base model
model_completion = model_completion.merge_and_unload()

# Save the model
model_completion.save_pretrained("ckpt-completion")

del model_completion
torch.cuda.empty_cache()
print("\nTask 1 complete. Model saved to ckpt-completion/ and freed from memory.")

Epoch 1/3 - Evaluating: 100%|██████████| 21/21 [00:24<00:00,  1.17s/it]


Epoch 1/3 - Average Train Loss: 1.5637, Average Dev Loss: 1.4651


Epoch 2/3 - Evaluating: 100%|██████████| 21/21 [00:27<00:00,  1.31s/it]


Epoch 2/3 - Average Train Loss: 1.3567, Average Dev Loss: 1.3885


Epoch 3/3 - Evaluating: 100%|██████████| 21/21 [00:26<00:00,  1.26s/it]


Epoch 3/3 - Average Train Loss: 1.2680, Average Dev Loss: 1.3502

Task 1 complete. Model saved to ckpt-completion/ and freed from memory.


# Task 2: Full-prompt tuning

Fine-tune using **all tokens** (instruction + response):

$$\mathcal{L}_\theta = -\frac{1}{T} \sum_{t=1}^{T} \log P(s_t \mid s_{<t})$$

Here the labels are identical to `input_ids` (except padding which is masked with `-100`).  
The model must learn to predict instruction tokens as well as response tokens.

### Task 2a: Tokenize full sequence in labels

In [16]:
def tokenize_full_prompt(sample: dict, tokenizer, max_length: int) -> dict[str, list[int]]:
    """
    Tokenize one dataset sample for Task 2 (full-prompt tuning).

    Unlike completion-only tuning, the labels here cover the entire token
    sequence s = [x; y]. The model is trained to predict every token,
    including instruction tokens.


    Args:
        sample: A single dataset record with keys 'instruction', 'input',
                 and 'output'.
        tokenizer: The tokenizer to use for encoding the text.
        max_length: Maximum sequence length for truncation and padding.

    Returns:
        A dict with three equal-length lists (length is max_length):
            input_ids      – token ids for the full sequence [x; y],
                             right-padded with pad_token_id.
            attention_mask – 1 for real tokens, 0 for padding positions.
            labels         – same as input_ids for all real token positions;
                             -100 at padding positions (ignored by the loss).
    """
    # TODO: your code here
    prompt, response = format_prompt(sample)
    tokens = tokenizer(prompt + response, truncation=True, max_length=max_length, padding="max_length")
    input_ids = tokens["input_ids"]
    attention_mask = tokens["attention_mask"]
    labels = []
    
    for i in range(len(input_ids)):
        if attention_mask[i] == 0:
            labels.append(-100)
        else:
            labels.append(input_ids[i])
            
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }


### Data Preparation

In [17]:
#TODO: Set MAX_LENGTH chosen before 
MAX_LENGTH = 512

# Prepare train/dev dataset
remove_cols = dataset["train"].column_names
tokenized_full = DatasetDict({
    split: dataset[split].map(
        tokenize_full_prompt,
        fn_kwargs={"tokenizer": tokenizer, "max_length": MAX_LENGTH},
        remove_columns=remove_cols,
    )
    for split in ("train", "dev")
})

# Convert to tensors
tokenized_full.set_format("torch")
print(tokenized_full)

Map: 100%|██████████| 164/164 [00:00<00:00, 2077.42 examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 767
    })
    dev: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 164
    })
})


### Task 2b: Fine-tune and save the full-prompt model

We reuse the same `load_model` and `train_model` function defined in Task 1.

In [18]:
model_full = load_model()

# TODO: Call train_model with the hyperparameters you chose
model_full = train_model(model_full, tokenized_full)

# Merge LoRA weights into base model
model_full = model_full.merge_and_unload()

model_full.save_pretrained("ckpt-full")
del model_full
torch.cuda.empty_cache()
print("\nTask 2 complete. Model saved to ckpt-full/ and freed from memory.")

Epoch 1/3 - Evaluating: 100%|██████████| 21/21 [00:29<00:00,  1.42s/it]


Epoch 1/3 - Average Train Loss: 2.4386, Average Dev Loss: 2.2707


Epoch 2/3 - Evaluating: 100%|██████████| 21/21 [00:26<00:00,  1.24s/it]


Epoch 2/3 - Average Train Loss: 2.1524, Average Dev Loss: 2.1703


Epoch 3/3 - Evaluating: 100%|██████████| 21/21 [00:26<00:00,  1.24s/it]


Epoch 3/3 - Average Train Loss: 2.0664, Average Dev Loss: 2.1147

Task 2 complete. Model saved to ckpt-full/ and freed from memory.


## Task 3: Evaluation with Perplexity

We report two perplexity metrics on the **test set** for both fine-tuned models:

$$\text{PPL}_{\text{resp}} = \exp\!\left(\frac{1}{n}\sum_{t=m+1}^{m+n} -\log P(s_t \mid s_{<t})\right)$$

$$\text{PPL}_{\text{all}}  = \exp\!\left(\frac{1}{T}\sum_{t=1}^{T}   -\log P(s_t \mid s_{<t})\right)$$

- $\text{PPL}_{\text{resp}}$ — how well the model predicts **response tokens only**.
- $\text{PPL}_{\text{all}}$  — how well the model predicts the **entire sequence** (instruction + response).

### Task 3a: Tokenize the Test set

In [21]:
from urllib import response


def tokenize_for_eval(sample: dict, tokenizer, max_length: int) -> dict[str, list[int] | int]:
    """
    Tokenize one test sample and record the instruction boundary for evaluation.

    Produces the same input_ids / attention_mask as tokenize_full_prompt, but
    also stores 'prompt_len' (m) so that compute_perplexities can separate
    instruction positions from response positions when computing PPL_resp.

    Args:
        sample: A single dataset record with keys 'instruction', 'input',
                 and 'output'.
        tokenizer: The tokenizer to use for encoding the text.
        max_length: Maximum sequence length for truncation and padding.

    Returns:
        A dict with:
            input_ids      – token ids for the full sequence [x; y],
                             right-padded to max_length.
            attention_mask – 1 for real tokens, 0 for padding positions.
            prompt_len     – m, the number of instruction tokens in input_ids
                             (before any response tokens begin).
    """
    # TODO: your code here
    prompt, response = format_prompt(sample)
    tokens = tokenizer(prompt + response, truncation=True, max_length=max_length, padding="max_length")
    input_ids = tokens["input_ids"]
    attention_mask = tokens["attention_mask"]
    
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    bos_offset = 1 if tokenizer.bos_token_id is not None else 0
    prompt_len = len(prompt_ids) + bos_offset
    
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "prompt_len": prompt_len
    }


### Test Data Preparation

In [22]:
#TODO: Set MAX_LENGTH chosen before 
MAX_LENGTH = 512

# Prepare test dataset
test_eval = dataset["test"].map(
    tokenize_for_eval,
    fn_kwargs={"tokenizer": tokenizer, "max_length": MAX_LENGTH},
    remove_columns=dataset["test"].column_names,
)
test_eval.set_format("torch")
print(f"Test examples: {len(test_eval)}")

Map: 100%|██████████| 166/166 [00:00<00:00, 1326.50 examples/s]

Test examples: 166


### Task 3b: Compute perplexity
We run a forward pass, compute per-token NLL with `F.cross_entropy(..., reduction='none')`, then accumulate separately over response positions and all positions.

In [25]:
@torch.no_grad()
def compute_perplexities(
    model: AutoModelForCausalLM,
    test_dataset: "datasets.Dataset",
    batch_size: int = 8,
) -> dict[str, float]:
    """
    Compute PPL_resp and PPL_all for a fine-tuned model on a test dataset.

    Both metrics are derived from per-token negative log-likelihoods (NLL)
    obtained via a single forward pass. The causal shift (logit at position t
    predicts token t+1) is applied manually so we have full control over which
    positions contribute to each metric.

    PPL_resp accumulates NLL only over response token positions t in (m, ..., T-1),
    implementing:
        PPL_resp = exp( 1/n * sum_{t=m+1}^{T} -log P(s_t | s_{<t}) )

    PPL_all accumulates NLL over all real (non-padding) token positions,
    implementing:
        PPL_all = exp( 1/T * sum_{t=1}^{T} -log P(s_t | s_{<t}) )

    NLLs are accumulated across the entire test set before taking the mean,
    so sequences of different lengths are weighted by token count rather than
    by sample count.

    Args:
        model:        A fine-tuned AutoModelForCausalLM to evaluate.
        test_dataset: A HuggingFace Dataset with columns:
                          input_ids      (LongTensor, shape T)
                          attention_mask (LongTensor, shape T)
                          prompt_len     (int) — number of instruction tokens m
        batch_size:   Number of samples processed per forward pass (default 8).

    Returns:
        A dict with two float entries:
            'ppl_resp' – perplexity over response tokens only.
            'ppl_all'  – perplexity over the full token sequence.

    Note:
        Be mindful of the causal shift when mapping prompt_len m to indices in
        the shifted sequence — shifted index i predicts original token i+1.
    """
    # TODO: your code here
    model.to(DEVICE)
    total_nll_all = 0.0
    total_tokens_all = 0
    total_nll_resp = 0.0
    total_tokens_resp = 0
    
    dataloader = DataLoader(test_dataset, batch_size=batch_size)
    
    for batch in tqdm(dataloader, desc="Evaluating"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        prompt_lens = batch["prompt_len"].to(DEVICE)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        
        # Casual shift
        shifted_logits = logits[:, :-1, :].contiguous()
        shifted_labels = input_ids[:, 1:].contiguous()
        shifted_attention_mask = attention_mask[:, 1:].float()
        
        B, T1, V = shifted_logits.shape
        
        nll = F.cross_entropy(
            shifted_logits.view(-1, V),
            shifted_labels.view(-1),
            reduction="none"
        ).view(B, T1)
        
        # PPL all
        total_nll_all += (nll * shifted_attention_mask).sum().item()
        total_tokens_all += shifted_attention_mask.sum().item()
        
        # PPL resp
        positions = torch.arange(T1, device=DEVICE).unsqueeze(0)
        resp_mask = (positions >= (prompt_lens -1).unsqueeze(1)).float() * shifted_attention_mask
        total_nll_resp += (nll * resp_mask).sum().item()
        total_tokens_resp += resp_mask.sum().item()
        
    ppl_all = math.exp(total_nll_all / total_tokens_all) if total_tokens_all > 0 else float("inf")
    ppl_resp = math.exp(total_nll_resp / total_tokens_resp) if total_tokens_resp > 0 else float("inf")
    
    return {"ppl_all": ppl_all, "ppl_resp": ppl_resp}


### Task 3c: Run evaluation on both fine-tuned models

In [26]:
print("Evaluating completion-only model (Task 1)...")
# Load the saved model
model_completion = AutoModelForCausalLM.from_pretrained("ckpt-completion")
results_completion = compute_perplexities(model_completion, test_eval)
del model_completion
torch.cuda.empty_cache()

print("\nEvaluating full-prompt model (Task 2)...")
# Load the saved model
model_full = AutoModelForCausalLM.from_pretrained("ckpt-full")
results_full = compute_perplexities(model_full, test_eval)
del model_full
torch.cuda.empty_cache()

print(f"\nCompletion-only: ppl_resp={results_completion['ppl_resp']:.2f}, ppl_all={results_completion['ppl_all']:.2f}")
print(f"Full-prompt:     ppl_resp={results_full['ppl_resp']:.2f}, ppl_all={results_full['ppl_all']:.2f}")

Evaluating completion-only model (Task 1)...


Evaluating: 100%|██████████| 21/21 [00:42<00:00,  2.01s/it]



Evaluating full-prompt model (Task 2)...


Evaluating: 100%|██████████| 21/21 [00:24<00:00,  1.16s/it]


Completion-only: ppl_resp=3.89, ppl_all=17.87
Full-prompt:     ppl_resp=4.14, ppl_all=7.99


## Interpretation of Results

Fill in your numbers after running the evaluation cell and interpret your results:

| Model | $\text{PPL}_{\text{resp}}$ | $\text{PPL}_{\text{all}}$ |
|-------|:--------------------------:|:-------------------------:|
| Qwen1.5-0.5B-completion-only | 3.89 | 17.87 |
| Qwen1.5-0.5B-full-prompt     | 4.14 | 7.99 |


### Interpreatation of Results

#### PPL_resp

Around 4 for both so the model is choosing among 4 equally likely tokens per response. My interpretation of that would be that both training strategies teach response generation (more or less) equally well

#### PPL_all

Here there is a big gap between the two models (8 vs 18). The completion model seems to be "surprised" by intruction tokens where the full prompt model isn't. Though i think that this is by desing.

#### Takeaway
- Both models learn to generate responses equally well: What I trained on during the response matters more than wheter i trained on the instruction as well
- Full prompt additional teaches the structure of the entire prompt format: can predict how instructions are phrased btter
- Full prompt gives richer representation and completion only is more focused